In [0]:
# Notebook: 02_generate_dark_stores
# Purpose: Simulate dark store master data for ZapFlow platform
# Layer: Simulation

import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F

# ── Configuration ─────────────────────────────────────────────
NUM_STORES = 50
OUTPUT_PATH = "/Volumes/zapflow_raw/raw_data/raw_files/dark_stores/"
SEED = 42
random.seed(SEED)

# ── Realistic dark store data ──────────────────────────────────
CITIES = ["Mumbai", "Delhi", "Bengaluru", "Hyderabad", "Chennai", "Pune", "Kolkata", "Ahmedabad"]

CITY_ZONES = {
    "Mumbai":    ["Andheri", "Bandra", "Powai", "Thane", "Borivali"],
    "Delhi":     ["Connaught Place", "Lajpat Nagar", "Dwarka", "Rohini", "Saket"],
    "Bengaluru": ["Koramangala", "Indiranagar", "Whitefield", "HSR Layout", "Marathahalli"],
    "Hyderabad": ["Banjara Hills", "Gachibowli", "Madhapur", "Kukatpally", "Secunderabad"],
    "Chennai":   ["Anna Nagar", "Velachery", "OMR", "Adyar", "Tambaram"],
    "Pune":      ["Kothrud", "Viman Nagar", "Baner", "Hinjewadi", "Wakad"],
    "Kolkata":   ["Salt Lake", "New Town", "Park Street", "Howrah", "Behala"],
    "Ahmedabad": ["Navrangpura", "Satellite", "Bodakdev", "Vastrapur", "Maninagar"]
}

STORE_STATUS = ["active", "active", "active", "inactive", "under_maintenance"]

# ── Helper ─────────────────────────────────────────────────────
def random_date(start_year=2021, end_year=2023):
    start = datetime(start_year, 1, 1)
    end   = datetime(end_year, 12, 31)
    return (start + timedelta(days=random.randint(0, (end - start).days))).date()

def introduce_data_quality_issues(record, store_id):
    issue_type = None
    if store_id % 20 == 0:
        record["capacity_sqft"] = None
        issue_type = "missing_capacity"
    elif store_id % 30 == 0:
        record["operational_since"] = str(datetime(2099, 1, 1).date())
        issue_type = "future_date"
    record["_data_quality_issue"] = issue_type
    return record

# ── Generate records ───────────────────────────────────────────
def generate_dark_stores(n):
    records = []
    store_counter = 1

    for city in CITIES:
        zones = CITY_ZONES[city]
        # Distribute stores across cities — bigger cities get more stores
        city_store_count = 8 if city in ["Mumbai", "Delhi", "Bengaluru"] else 4

        for zone in zones[:city_store_count // len(zones) + 1]:
            if store_counter > n:
                break

            store_id = f"STORE_{store_counter:04d}"
            capacity = random.randint(1500, 5000)

            record = {
                "store_id":           store_id,
                "store_name":         f"ZapFlow {zone}",
                "city":               city,
                "zone":               zone,
                "address":            f"{random.randint(1, 200)}, {zone} Main Road",
                "pincode":            f"{random.randint(100000, 999999)}",
                "capacity_sqft":      capacity,
                "max_orders_per_day": capacity // 10,
                "operational_since":  str(random_date()),
                "store_status":       random.choice(STORE_STATUS),
                "delivery_radius_km": round(random.uniform(1.5, 4.0), 1),
                "_data_quality_issue": None
            }

            record = introduce_data_quality_issues(record, store_counter)
            records.append(record)
            store_counter += 1

    return records

# ── Main execution ─────────────────────────────────────────────
print("Generating dark store data...")
store_records = generate_dark_stores(NUM_STORES)

schema = StructType([
    StructField("store_id",            StringType(),  False),
    StructField("store_name",          StringType(),  True),
    StructField("city",                StringType(),  True),
    StructField("zone",                StringType(),  True),
    StructField("address",             StringType(),  True),
    StructField("pincode",             StringType(),  True),
    StructField("capacity_sqft",       IntegerType(), True),
    StructField("max_orders_per_day",  IntegerType(), True),
    StructField("operational_since",   StringType(),  True),
    StructField("store_status",        StringType(),  True),
    StructField("delivery_radius_km",  DoubleType(),  True),
    StructField("_data_quality_issue", StringType(),  True),
])

df_stores = spark.createDataFrame(store_records, schema=schema)

# ── Save to Volume ─────────────────────────────────────────────
(df_stores
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(OUTPUT_PATH))

print(f"✅ Generated {len(store_records)} dark store records")
print(f"✅ Saved to {OUTPUT_PATH}")

# ── Sanity checks ──────────────────────────────────────────────
print("\n── Stores per city ──")
df_stores.groupBy("city").count().orderBy("count", ascending=False).show()

print("── Store status distribution ──")
df_stores.groupBy("store_status").count().show()

print("── Data quality issues ──")
df_stores.groupBy("_data_quality_issue").count().show()

print("── Sample records ──")
df_stores.show(5, truncate=False)